# Поиск по фото

Тот же запуск, что из корня репозитория:

- энкодер `weights/finetuned_v4/encoder.onnx`
- индекс `weights/finetuned_v4/books_index.npz`
- эталон `../artifacts_clean/books`
- top-3
- детектор из `configs/detection/detection_config.yaml`

Можно отметить сколько угодно снимков. Top-k задаётся числом. Картинки одного размера и идут сеткой в несколько рядов.

In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "pipeline.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Запускайте ноутбук из репозитория CeramicNet")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.detection.detector import StampDetector
from src.paths import CONFIGS_DIR, REPORTS_DIR
from src.pipeline import CeramicPipeline
from src.retrieval.encoder import StampEncoder
from src.retrieval.index import StampIndex
from src.utils import load_yaml

ENCODER = ROOT / "weights/finetuned_v4/encoder.onnx"
INDEX = ROOT / "weights/finetuned_v4/books_index.npz"
BOOKS = (ROOT.parent / "artifacts_clean" / "books").resolve()
TOP_K = 3
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg"}


def find_stamps(root: Path) -> Path:
    for base in (root, root.parent):
        if not base.is_dir():
            continue
        for child in sorted(base.iterdir(), key=lambda path: path.name):
            stamps = child / "stamps"
            if stamps.is_dir() and any(stamps.glob("*.png")):
                return stamps
    raise FileNotFoundError("Папка stamps не найдена рядом с репозиторием")


def list_images(folder: Path) -> list[Path]:
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )


for path, title in ((ENCODER, "энкодер"), (INDEX, "индекс"), (BOOKS, "эталон")):
    if not path.exists():
        raise FileNotFoundError(f"Не найден {title}: {path}")

ret_cfg = load_yaml(CONFIGS_DIR / "retrieval" / "retrieval_config.yaml")
det_cfg = load_yaml(CONFIGS_DIR / "detection" / "detection_config.yaml")

encoder = StampEncoder.from_mapping(ret_cfg["encoder"], weights=ENCODER)
index = StampIndex.load(INDEX, books_dir=BOOKS)

module_path, class_name = det_cfg["processor"].rsplit(".", 1)
processor_cls = getattr(importlib.import_module(module_path), class_name)
processor = processor_cls(
    imgsz=det_cfg.get("imgsz", 640),
    conf_threshold=det_cfg.get("conf_threshold", 0.7),
    iou_threshold=det_cfg.get("iou_threshold", 0.5),
)
detector = StampDetector(
    model_path=ROOT / det_cfg["weights"],
    processor=processor,
)
pipeline = CeramicPipeline(
    encoder=encoder,
    index=index,
    detector=detector,
    conf_threshold=float(det_cfg.get("conf_threshold", 0.75)),
    top_k=TOP_K,
)

STAMPS_DIR = find_stamps(ROOT)
PHOTOS = list_images(STAMPS_DIR)
if not PHOTOS:
    raise FileNotFoundError(f"В папке нет изображений: {STAMPS_DIR}")

print(f"Фото в папке: {len(PHOTOS)}")
print(f"Папка: {STAMPS_DIR}")
print(f"Энкодер: {ENCODER.name}")
print(f"Индекс: {INDEX.name}, эталонов {len(index.records)}")
print(f"Детектор: {Path(det_cfg['weights']).name}, порог {pipeline.conf_threshold}")


Строка сужает список. Отметьте строки с Cmd или Shift и жмите «Искать». Чтобы собрать снимки из разных фильтров, добавляйте их кнопкой «Добавить». «Свои файлы» берут сколько угодно картинок с диска.

In [ ]:
import base64
import io

import cv2
from IPython.display import HTML
from PIL import Image as PILImage
from PIL import ImageDraw, ImageFont

from src.preprocessing.crop import crop_stamp
from src.preprocessing.image import read_bgr

TILE = 280
PER_ROW = 4
GAP = 10
HEADER = 72

try:
    LABEL_FONT = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial.ttf", 16)
except OSError:
    LABEL_FONT = ImageFont.load_default()

names = [path.name for path in PHOTOS]
by_name = {path.name: path for path in PHOTOS}
chosen: list[Path] = []

query = widgets.Text(
    placeholder="часть имени или путь к файлу",
    description="Найти",
    layout=widgets.Layout(width="92%"),
)
picker = widgets.SelectMultiple(
    options=names,
    rows=14,
    layout=widgets.Layout(width="92%", height="320px"),
)
chosen_box = widgets.SelectMultiple(
    options=[],
    rows=6,
    layout=widgets.Layout(width="92%", height="160px"),
)
add_button = widgets.Button(description="Добавить")
remove_button = widgets.Button(description="Убрать")
clear_button = widgets.Button(description="Очистить")
upload = widgets.FileUpload(
    accept=".png,.jpg,.jpeg",
    multiple=True,
    description="Свои файлы",
)
top_k = widgets.IntText(value=TOP_K, description="top-k", layout=widgets.Layout(width="160px"))
detect = widgets.Checkbox(value=True, description="детектор")
button = widgets.Button(description="Искать", button_style="primary")
output = widgets.Output()


def resolve_query(text: str) -> Path:
    raw = text.strip().strip('"').strip("'")
    if not raw:
        raise ValueError("Выберите фото или вставьте путь")
    if raw in by_name:
        return by_name[raw]

    path = Path(raw).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend((ROOT / raw, STAMPS_DIR / raw))
    for candidate in candidates:
        if candidate.is_file():
            return candidate

    needle = raw.lower()
    hits = [path for path in PHOTOS if needle in path.name.lower()]
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        shown = ", ".join(path.name for path in hits[:8])
        raise ValueError(f"Подходит несколько файлов: {shown}")
    raise FileNotFoundError(raw)


def chosen_label(path: Path) -> str:
    if path.parent == STAMPS_DIR:
        return path.name
    return str(path)


def refresh_chosen() -> None:
    chosen_box.options = [chosen_label(path) for path in chosen]


def on_filter(change) -> None:
    raw = change["new"].strip().lower()
    if not raw:
        picker.options = names
        return
    matched = [name for name in names if raw in name.lower()]
    picker.options = matched or names


def on_add(_) -> None:
    added = [by_name[name] for name in picker.value if name in by_name]
    if not added and query.value.strip():
        try:
            added = [resolve_query(query.value)]
        except (OSError, ValueError) as exc:
            with output:
                clear_output(wait=True)
                print(exc)
            return
    known = {path.resolve() for path in chosen if path.exists()}
    for path in added:
        key = path.resolve() if path.exists() else path
        if key in known:
            continue
        known.add(key)
        chosen.append(path)
    refresh_chosen()


def on_remove(_) -> None:
    drop = set(chosen_box.value)
    chosen[:] = [path for path in chosen if chosen_label(path) not in drop]
    refresh_chosen()


def on_clear(_) -> None:
    chosen.clear()
    refresh_chosen()


def take_uploads() -> list[Path]:
    value = upload.value
    if not value:
        return []
    saved = []
    for index, item in enumerate(value):
        if isinstance(item, dict):
            name, content = item["name"], item["content"]
        else:
            name, content = item.name, item.content
        dest = REPORTS_DIR / "_picked" / f"{index}_{Path(name).name}"
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(bytes(content))
        saved.append(dest)
    upload.value = ()
    return saved


def collect_paths() -> list[Path]:
    paths: list[Path] = []
    seen: set[str] = set()

    def push(path: Path) -> None:
        key = str(path.resolve()) if path.exists() else str(path)
        if key in seen:
            return
        seen.add(key)
        paths.append(path)

    for path in chosen:
        push(path)
    for name in picker.value:
        if name in by_name:
            push(by_name[name])
    for path in take_uploads():
        push(path)
    if not paths and query.value.strip():
        push(resolve_query(query.value))
    if not paths:
        raise ValueError("Отметьте фото или загрузите файлы")
    return paths


def wrap_text(text: str, limit: int = 22) -> list[str]:
    words = text.split()
    lines: list[str] = []
    current = ""
    for word in words:
        trial = word if not current else f"{current} {word}"
        if len(trial) <= limit:
            current = trial
            continue
        if current:
            lines.append(current)
        current = word
    if current:
        lines.append(current)
    return lines[:3]


def fit_tile(image: PILImage.Image, side: int) -> PILImage.Image:
    image = image.copy()
    image.thumbnail((side, side - 8))
    canvas = PILImage.new("RGB", (side, side), (248, 248, 248))
    canvas.paste(image, ((side - image.width) // 2, (side - image.height) // 2))
    return canvas


def labeled_tile(image: PILImage.Image, title: str) -> PILImage.Image:
    body = fit_tile(image, TILE)
    canvas = PILImage.new("RGB", (TILE, TILE + HEADER), (255, 255, 255))
    draw = ImageDraw.Draw(canvas)
    y = 4
    for line in wrap_text(title):
        draw.text((6, y), line, fill=(20, 20, 20), font=LABEL_FONT)
        y += 22
    canvas.paste(body, (0, HEADER))
    return canvas


def pack_tiles(tiles: list[PILImage.Image]) -> PILImage.Image:
    rows = [tiles[start:start + PER_ROW] for start in range(0, len(tiles), PER_ROW)]
    row_h = tiles[0].height
    width = PER_ROW * TILE + (PER_ROW - 1) * GAP
    height = len(rows) * row_h + (len(rows) - 1) * GAP
    sheet = PILImage.new("RGB", (width, height), (255, 255, 255))
    for row_index, row in enumerate(rows):
        for col_index, tile in enumerate(row):
            x = col_index * (TILE + GAP)
            y = row_index * (row_h + GAP)
            sheet.paste(tile, (x, y))
    return sheet


def show_sheet(sheet: PILImage.Image) -> None:
    buffer = io.BytesIO()
    sheet.save(buffer, format="JPEG", quality=90)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    display(HTML(
        f'<img src="data:image/jpeg;base64,{encoded}" '
        'style="max-width:100%;height:auto">'
    ))


def show_result(result: dict, image_path: Path) -> None:
    image_bgr = read_bgr(image_path)
    detections = result.get("detections", [])
    if not detections:
        detections = [{"box": None, "score": None, "matches": []}]
    print(f"{image_path.name}  детекций: {len(result.get('detections', []))}")

    for number, detection in enumerate(detections, start=1):
        score = detection["score"]
        box = detection["box"]
        matches = detection.get("matches", [])
        score_text = f"{score:.3f}" if score is not None else "без детектора"
        print(f"  #{number}  conf {score_text}")
        for match in matches:
            factory = match.get("factory") or ""
            print(
                f"    {match['rank']}. {match['score']:.4f}  "
                f"{match['mark_id']}  {match.get('book') or ''}  {factory}"
            )

        photo = PILImage.fromarray(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
        if box is not None and score is not None:
            draw = ImageDraw.Draw(photo)
            width = max(photo.size)
            draw.rectangle(
                tuple(int(v) for v in box),
                outline=(210, 35, 35),
                width=max(4, width // 80),
            )
            photo_title = f"Детекция {score:.2f}"
            crop_bgr = crop_stamp(
                image_bgr,
                box,
                padding_ratio=0.0,
                target_size=pipeline.target_size,
            )
        else:
            photo_title = "Фото целиком"
            height, width = image_bgr.shape[:2]
            crop_bgr = crop_stamp(
                image_bgr,
                (0.0, 0.0, float(width), float(height)),
                padding_ratio=0.0,
                target_size=pipeline.target_size,
            )
        crop = PILImage.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        tiles = [
            labeled_tile(photo, photo_title),
            labeled_tile(crop, "Кроп"),
        ]
        for match in matches:
            match_path = Path(match["image_path"])
            if match_path.is_file():
                mark = PILImage.open(match_path).convert("RGB")
            else:
                mark = PILImage.new("RGB", (TILE, TILE), (230, 230, 230))
            factory = match.get("factory") or ""
            title = f"{match['rank']}. {match['score']:.3f} {match['mark_id']} {factory}"
            tiles.append(labeled_tile(mark, title))
        show_sheet(pack_tiles(tiles))


def on_search(_):
    with output:
        clear_output(wait=True)
        try:
            image_paths = collect_paths()
        except (OSError, ValueError) as exc:
            print(exc)
            return
        print(f"Фото: {len(image_paths)}")
        k = int(top_k.value)
        if k < 1:
            print("top-k должен быть целым числом от 1")
            return
        for image_path in image_paths:
            result = pipeline.predict(
                image_path,
                top_k=k,
                detect=bool(detect.value),
            )
            show_result(result, image_path)


query.observe(on_filter, names="value")
add_button.on_click(on_add)
remove_button.on_click(on_remove)
clear_button.on_click(on_clear)
button.on_click(on_search)
display(widgets.VBox([
    query,
    picker,
    widgets.HBox([add_button, remove_button, clear_button]),
    widgets.Label("Накоплено"),
    chosen_box,
    widgets.HBox([upload, top_k, detect, button]),
    output,
]))
